In [4]:
"""
增强版训练脚本 - 适用于Colab长时间训练
修复eval_strategy错误，并将部分模型保存到Colab本地磁盘
"""

import os
import sys
import json
import math
import time
import torch
import signal
import logging
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

# Transformers库
from transformers import (
    GPT2Tokenizer, GPT2LMHeadModel,
    TextDataset, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, TrainerCallback
)

# ========== Colab环境检测和配置 ==========
IS_COLAB = 'google.colab' in sys.modules
print(f"运行环境: {'Google Colab' if IS_COLAB else '本地环境'}")

if IS_COLAB:
    # 在Colab中安装必要的包
    !pip install transformers[torch] datasets accelerate -q
    !pip install matplotlib tqdm -q

    # 自动挂载Google Drive
    from google.colab import drive
    drive_mounted = False

    try:
        # 尝试强制重新挂载，避免"already mounted"错误
        drive.mount('/content/drive', force_remount=True)
        drive_mounted = True
        print("✓ Google Drive 挂载成功")
    except Exception as e:
        print(f"Drive挂载失败: {e}")
        print("将使用Colab临时存储")

# ========== 配置部分 ==========
class TrainingConfig:
    """训练配置类 - 部分模型保存到Colab本地磁盘"""

    # 基础路径配置
    if IS_COLAB:
        # Colab环境使用Google Drive
        BASE_DIR = "/content/drive/MyDrive/LLM_Impossible_Training"
        # Colab本地存储目录
        LOCAL_BASE_DIR = "/content/LLM_Impossible_Training"
    else:
        # 本地环境
        BASE_DIR = "./impossible_language_training"
        LOCAL_BASE_DIR = "./impossible_language_training_local"

    # 数据路径（始终在Google Drive）
    DATA_DIR = os.path.join(BASE_DIR, "data")

    # 结果保存路径（部分在Google Drive，部分在本地）
    RESULTS_DIR = os.path.join(BASE_DIR, "results")
    LOCAL_RESULTS_DIR = os.path.join(LOCAL_BASE_DIR, "results")

    # 检查点保存路径
    CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
    LOCAL_CHECKPOINT_DIR = os.path.join(LOCAL_BASE_DIR, "checkpoints")

    # 日志文件路径（始终在Google Drive）
    LOG_FILE = os.path.join(BASE_DIR, "training.log")

    # 训练配置
    BATCH_SIZE = 4
    NUM_EPOCHS = 5
    LEARNING_RATE = 5e-5
    SAVE_STEPS = 20  # 每20步保存一次
    LOGGING_STEPS = 5
    MAX_CHECKPOINTS = 3  # 最多保留3个检查点

    # 数据集配置
    # 前两个模型保存到Google Drive，后两个保存到Colab本地
    DATASETS = [
        {
            "name": "Natural Language",
            "file_path": os.path.join(DATA_DIR, "data_natural.txt"),
            "model_dir": os.path.join(RESULTS_DIR, "model_natural"),
            "checkpoint_dir": os.path.join(CHECKPOINT_DIR, "natural_language"),
            "storage_location": "drive",  # Google Drive
            "resume_checkpoint": None
        },
        {
            "name": "Impossible Language (Fixed Distance)",
            "file_path": os.path.join(DATA_DIR, "impossible_output_fixed_distance.txt"),
            "model_dir": os.path.join(RESULTS_DIR, "model_fixed_distance"),
            "checkpoint_dir": os.path.join(CHECKPOINT_DIR, "fixed_distance"),
            "storage_location": "drive",  # Google Drive
            "resume_checkpoint": None
        },
        {
            "name": "Impossible Language (Reversed)",
            "file_path": os.path.join(DATA_DIR, "impossible_output_reversed.txt"),
            "model_dir": os.path.join(LOCAL_RESULTS_DIR, "model_reversed"),
            "checkpoint_dir": os.path.join(LOCAL_CHECKPOINT_DIR, "reversed"),
            "storage_location": "local",  # Colab本地
            "resume_checkpoint": None
        },
        {
            "name": "Impossible Language (Parity Negation)",
            "file_path": os.path.join(DATA_DIR, "impossible_output_parity_negation.txt"),
            "model_dir": os.path.join(LOCAL_RESULTS_DIR, "model_parity_negation"),
            "checkpoint_dir": os.path.join(LOCAL_CHECKPOINT_DIR, "parity_negation"),
            "storage_location": "local",  # Colab本地
            "resume_checkpoint": None
        }
    ]

    @classmethod
    def create_directories(cls):
        """创建所有必要的目录"""
        # Google Drive上的目录
        drive_directories = [
            cls.BASE_DIR,
            cls.DATA_DIR,
            cls.RESULTS_DIR,
            cls.CHECKPOINT_DIR
        ]

        # Colab本地目录
        local_directories = [
            cls.LOCAL_BASE_DIR,
            cls.LOCAL_RESULTS_DIR,
            cls.LOCAL_CHECKPOINT_DIR
        ]

        print("创建Google Drive目录:")
        for directory in drive_directories:
            os.makedirs(directory, exist_ok=True)
            print(f"  ✓ {directory}")

        print("\n创建Colab本地目录:")
        for directory in local_directories:
            os.makedirs(directory, exist_ok=True)
            print(f"  ✓ {directory}")

        # 为每个数据集创建具体目录
        print("\n创建数据集目录:")
        for dataset in cls.DATASETS:
            os.makedirs(dataset["model_dir"], exist_ok=True)
            os.makedirs(dataset["checkpoint_dir"], exist_ok=True)
            location = "Google Drive" if dataset["storage_location"] == "drive" else "Colab本地"
            print(f"  ✓ {dataset['name']}: {dataset['model_dir']} ({location})")

# ========== 保持活跃管理器 ==========
class KeepAliveManager:
    """管理Colab会话的活跃状态"""

    @staticmethod
    def simulate_activity():
        """模拟用户活动，防止Colab断开连接"""
        try:
            # 生成一些输出（保持活跃）
            print(f"[保持活跃] {datetime.now().strftime('%H:%M:%S')} - 训练仍在进行中...")

            # 轻微的内存操作
            dummy_var = [random.random() for _ in range(1000)]
            del dummy_var

            # 检查GPU内存使用情况
            if torch.cuda.is_available():
                gpu_memory = torch.cuda.memory_allocated() / 1024**3
                print(f"[GPU内存] 已使用: {gpu_memory:.2f} GB")

            return True
        except:
            return False

# ========== 日志配置 ==========
def setup_logging(log_file):
    """设置日志记录"""
    # 确保日志文件所在目录存在
    log_dir = os.path.dirname(log_file)
    if log_dir and not os.path.exists(log_dir):
        os.makedirs(log_dir, exist_ok=True)

    # 创建日志记录器
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)

    # 移除现有的处理器，避免重复
    if logger.handlers:
        logger.handlers.clear()

    # 文件处理器
    file_handler = logging.FileHandler(log_file, encoding='utf-8')
    file_handler.setLevel(logging.INFO)

    # 控制台处理器
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)

    # 格式化器
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    console_handler.setFormatter(formatter)

    # 添加处理器
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    return logger

# ========== 自定义回调函数 ==========
class RobustColabCallback(TrainerCallback):
    """Colab稳健训练回调函数"""

    def __init__(self, trainer, checkpoint_dir, model_dir, logger=None):
        self.trainer = trainer
        self.checkpoint_dir = checkpoint_dir
        self.model_dir = model_dir
        self.logger = logger
        self.perplexities = []
        self.start_time = time.time()
        self.last_save_time = time.time()
        self.last_keepalive_time = time.time()
        self.checkpoint_counter = 0

        logger.info(f"稳健回调初始化，检查点目录: {checkpoint_dir}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        """记录日志时触发"""
        if logs and 'loss' in logs:
            # 计算并记录perplexity
            loss = logs['loss']
            perplexity = math.exp(loss) if loss < 100 else float('inf')
            logs['perplexity'] = perplexity
            self.perplexities.append(perplexity)

            # 定期模拟活动，防止断开
            current_time = time.time()
            if current_time - self.last_keepalive_time > 300:  # 每5分钟
                KeepAliveManager.simulate_activity()
                self.last_keepalive_time = current_time

            # 更频繁地保存检查点（每10分钟）
            if current_time - self.last_save_time > 600:  # 每10分钟
                self._emergency_save(state)
                self.last_save_time = current_time

    def on_step_end(self, args, state, control, **kwargs):
        """每个训练步结束时触发"""
        # 每50步保存一次
        if state.global_step % 50 == 0:
            self._emergency_save(state)

    def on_epoch_end(self, args, state, control, **kwargs):
        """每个epoch结束时触发"""
        if self.logger:
            self.logger.info(f"Epoch {state.epoch} 完成，总步数: {state.global_step}")
        self._save_checkpoint(state, is_epoch_end=True)

    def on_train_end(self, args, state, control, **kwargs):
        """训练结束时触发"""
        if self.logger:
            self.logger.info("训练完成，保存最终模型...")
        self._save_checkpoint(state, final_save=True)

    def _emergency_save(self, state):
        """紧急保存，最小化数据丢失"""
        try:
            # 简化保存，只保存最关键的信息
            save_dir = os.path.join(self.checkpoint_dir, f"emergency_{int(time.time())}")
            os.makedirs(save_dir, exist_ok=True)

            # 保存状态
            state_info = {
                "global_step": state.global_step,
                "epoch": state.epoch,
                "save_time": datetime.now().isoformat(),
                "perplexities": self.perplexities[-100:] if self.perplexities else []
            }

            with open(os.path.join(save_dir, "emergency_state.json"), 'w') as f:
                json.dump(state_info, f, indent=2)

            # 如果可能，保存模型
            try:
                self.trainer.save_model(save_dir)
            except:
                pass  # 如果保存模型失败，至少保存了状态

            if self.logger:
                self.logger.info(f"紧急检查点保存到: {save_dir}")

        except Exception as e:
            if self.logger:
                self.logger.error(f"紧急保存失败: {e}")

    def _save_checkpoint(self, state, is_epoch_end=False, final_save=False):
        """完整保存检查点"""
        try:
            if final_save:
                checkpoint_path = os.path.join(self.model_dir, "final_model")
                self.trainer.save_model(checkpoint_path)
                self.logger.info(f"最终模型保存到: {checkpoint_path}")
            else:
                checkpoint_path = os.path.join(
                    self.checkpoint_dir,
                    f"checkpoint-{state.global_step:06d}"
                )
                self.trainer.save_model(checkpoint_path)

                # 保存训练状态
                state_dict = {
                    "global_step": state.global_step,
                    "epoch": state.epoch,
                    "perplexities": self.perplexities,
                    "save_time": datetime.now().isoformat(),
                    "total_training_time": time.time() - self.start_time
                }

                state_file = os.path.join(checkpoint_path, "training_state.json")
                with open(state_file, 'w') as f:
                    json.dump(state_dict, f, indent=2)

                self.checkpoint_counter += 1

                if is_epoch_end:
                    self.logger.info(f"Epoch结束检查点保存到: {checkpoint_path}")
                else:
                    self.logger.info(f"检查点保存到: {checkpoint_path}")

        except Exception as e:
            self.logger.error(f"保存检查点时出错: {e}")

# ========== 训练函数 - 修复eval_strategy错误 ==========
def train_with_maximum_robustness(config, dataset_config, model_name, tokenizer, logger):
    """最大稳健性的训练函数"""

    logger.info(f"开始训练: {model_name}")
    logger.info(f"数据文件: {dataset_config['file_path']}")
    logger.info(f"模型保存到: {dataset_config['model_dir']} ({dataset_config['storage_location']})")

    # 检查数据文件是否存在
    if not os.path.exists(dataset_config['file_path']):
        logger.error(f"数据文件不存在: {dataset_config['file_path']}")
        return None, None, None

    # 创建模型输出目录
    model_dir = dataset_config['model_dir']
    os.makedirs(model_dir, exist_ok=True)

    # 检查是否有可恢复的检查点
    resume_from_checkpoint = None

    # 优先检查模型目录中的检查点
    if os.path.exists(model_dir):
        checkpoint_dirs = [d for d in os.listdir(model_dir) if d.startswith("checkpoint")]
        if checkpoint_dirs:
            # 选择最新的检查点
            latest_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))
            resume_from_checkpoint = os.path.join(model_dir, latest_checkpoint)
            logger.info(f"发现模型目录中的检查点: {resume_from_checkpoint}")

    # 检查指定的恢复检查点
    if dataset_config.get('resume_checkpoint'):
        specified_checkpoint = dataset_config['resume_checkpoint']
        if os.path.exists(specified_checkpoint):
            resume_from_checkpoint = specified_checkpoint
            logger.info(f"使用指定的检查点: {resume_from_checkpoint}")

    # 加载模型
    try:
        if resume_from_checkpoint and os.path.exists(resume_from_checkpoint):
            model = GPT2LMHeadModel.from_pretrained(resume_from_checkpoint)
            logger.info(f"从检查点恢复训练: {resume_from_checkpoint}")

            # 尝试加载训练状态
            state_file = os.path.join(resume_from_checkpoint, "training_state.json")
            if os.path.exists(state_file):
                with open(state_file, 'r') as f:
                    state_info = json.load(f)
                logger.info(f"恢复训练状态: 步数={state_info.get('global_step', 0)}, epoch={state_info.get('epoch', 0)}")
        else:
            model = GPT2LMHeadModel.from_pretrained("gpt2")
            logger.info("创建新的GPT-2模型")
    except Exception as e:
        logger.error(f"加载模型失败: {e}，创建新模型")
        model = GPT2LMHeadModel.from_pretrained("gpt2")

    # 准备数据集
    try:
        train_dataset = TextDataset(
            tokenizer=tokenizer,
            file_path=dataset_config['file_path'],
            block_size=128
        )
        logger.info(f"数据集加载成功，样本数: {len(train_dataset)}")
    except Exception as e:
        logger.error(f"加载数据集失败: {e}")
        return None, None, None

    # 数据收集器
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )

    # 训练参数 - 修复eval_strategy错误
    training_args = TrainingArguments(
        output_dir=model_dir,
        overwrite_output_dir=True,
        num_train_epochs=config.NUM_EPOCHS,
        per_device_train_batch_size=config.BATCH_SIZE,
        logging_steps=config.LOGGING_STEPS,
        save_steps=config.SAVE_STEPS,
        save_total_limit=config.MAX_CHECKPOINTS,
        learning_rate=config.LEARNING_RATE,
        weight_decay=0.01,
        adam_epsilon=1e-8,
        max_grad_norm=1.0,
        warmup_steps=100,
        save_strategy="steps",
        load_best_model_at_end=False,
        report_to="none",
        push_to_hub=False,
        remove_unused_columns=False,
        gradient_accumulation_steps=1,
        dataloader_num_workers=0,
        disable_tqdm=False,
        logging_first_step=True,
        logging_dir=os.path.join(model_dir, "logs"),
        # 修复：因为我们不需要评估，所以将eval_strategy设置为"no"
        eval_strategy="no",  # 设置为"no"而不是"steps"
    )

    # 创建检查点目录
    checkpoint_dir = dataset_config['checkpoint_dir']
    os.makedirs(checkpoint_dir, exist_ok=True)

    # 创建Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )

    # 添加稳健回调
    robust_callback = RobustColabCallback(trainer, checkpoint_dir, model_dir, logger=logger)

    # 开始训练
    try:
        logger.info("开始训练过程...")
        logger.info("=" * 60)
        logger.info("重要提示：")
        logger.info(f"1. 模型将保存到: {model_dir}")
        logger.info(f"2. 检查点将每{config.SAVE_STEPS}步保存一次")
        logger.info("3. 每10分钟会有紧急保存")
        logger.info("4. 如果Colab断开，可以从最新检查点恢复")
        logger.info("=" * 60)

        train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)

        # 保存最终模型
        trainer.save_model()
        tokenizer.save_pretrained(model_dir)

        # 收集训练指标
        log_history = trainer.state.log_history
        losses = [x['loss'] for x in log_history if 'loss' in x]
        perplexities = robust_callback.perplexities

        logger.info(f"训练完成: {model_name}")
        if losses:
            logger.info(f"最终损失: {losses[-1]:.4f}")
            logger.info(f"最小损失: {min(losses):.4f}")
        if perplexities:
            logger.info(f"最终困惑度: {perplexities[-1]:.4f}")

        return losses, perplexities, train_result

    except KeyboardInterrupt:
        logger.warning(f"训练被用户中断: {model_name}")
        # 紧急保存
        robust_callback._emergency_save(trainer.state)
        return None, None, None

    except Exception as e:
        logger.error(f"训练过程中出错: {e}")
        import traceback
        logger.error(traceback.format_exc())

        # 尝试紧急保存
        try:
            robust_callback._emergency_save(trainer.state)
        except:
            pass

        return None, None, None

# ========== 主函数 ==========
def main():
    """主训练函数"""

    print("=" * 80)
    print("Colab稳健训练脚本 - 不可能语言实验")
    print("=" * 80)
    print("存储策略：")
    print("1. Natural Language: Google Drive")
    print("2. Fixed Distance: Google Drive")
    print("3. Reversed: Colab本地磁盘")
    print("4. Parity Negation: Colab本地磁盘")
    print("=" * 80)

    # 重要提示
    if IS_COLAB:
        print("\n⚠️  存储位置说明：")
        print("   • Natural Language 和 Fixed Distance 模型保存到 Google Drive")
        print("   • Reversed 和 Parity Negation 模型保存到 Colab本地磁盘")
        print("   • 本地存储的数据在Colab会话结束后会丢失！")
        print("   • 如需保留本地模型，请在训练完成后手动复制到Google Drive")
        print("=" * 80)

    # 创建目录结构
    config = TrainingConfig
    config.create_directories()

    # 初始化日志记录器
    logger = setup_logging(config.LOG_FILE)
    logger.info("训练脚本开始执行 - 混合存储版")
    logger.info(f"Google Drive工作目录: {config.BASE_DIR}")
    logger.info(f"Colab本地工作目录: {config.LOCAL_BASE_DIR}")

    # 初始化分词器
    logger.info("初始化分词器...")
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # 训练所有数据集
    all_metrics = {}

    for dataset_config in config.DATASETS:
        dataset_name = dataset_config['name']
        storage_location = dataset_config['storage_location']

        print(f"\n{'='*60}")
        print(f"训练数据集: {dataset_name}")
        print(f"存储位置: {'Google Drive' if storage_location == 'drive' else 'Colab本地磁盘'}")
        print(f"{'='*60}")

        # 检查是否需要跳过（如果已经有结果）
        result_file = os.path.join(dataset_config['model_dir'], "training_results.json")
        if os.path.exists(result_file):
            logger.info(f"发现已有训练结果: {result_file}")
            choice = input(f"数据集 '{dataset_name}' 已有训练结果，是否重新训练？(y/N): ")
            if choice.lower() != 'y':
                logger.info(f"跳过数据集: {dataset_name}")

                # 加载已有结果
                try:
                    with open(result_file, 'r') as f:
                        existing_results = json.load(f)
                    all_metrics[dataset_name] = existing_results
                except:
                    pass
                continue

        # 开始训练计时
        start_time = time.time()

        # 训练模型（使用稳健版本）
        losses, perplexities, train_result = train_with_maximum_robustness(
            config, dataset_config, dataset_name, tokenizer, logger
        )

        # 计算训练时间
        training_time = time.time() - start_time

        if losses is not None:
            # 保存训练结果
            metrics = {
                "losses": losses,
                "perplexities": perplexities,
                "final_loss": losses[-1] if losses else None,
                "training_time": training_time,
                "total_steps": len(losses),
                "dataset": dataset_name,
                "storage_location": storage_location,
                "file_path": dataset_config['file_path'],
                "model_dir": dataset_config['model_dir'],
                "completed_at": datetime.now().isoformat()
            }

            all_metrics[dataset_name] = metrics

            # 保存结果到文件
            result_file = os.path.join(dataset_config['model_dir'], "training_results.json")
            with open(result_file, 'w') as f:
                json.dump(metrics, f, indent=2)

            logger.info(f"训练结果保存到: {result_file}")

            # 打印训练统计
            print(f"\n✅ 训练完成 - {dataset_name}:")
            print(f"   存储位置: {'Google Drive' if storage_location == 'drive' else 'Colab本地磁盘'}")
            print(f"   训练步数: {len(losses)}")
            print(f"   最终损失: {losses[-1]:.4f}" if losses else "   最终损失: N/A")
            print(f"   训练时间: {training_time/60:.2f} 分钟")

            # 如果是本地存储，提醒用户备份
            if storage_location == "local" and IS_COLAB:
                print(f"\n⚠️  重要提醒：此模型保存在Colab本地磁盘")
                print(f"   路径: {dataset_config['model_dir']}")
                print(f"   Colab会话结束后这些数据会丢失！")
                print(f"   请在会话结束前手动备份到Google Drive")

                # 提供备份命令
                drive_backup_dir = os.path.join(config.RESULTS_DIR, f"backup_{dataset_name.replace(' ', '_').lower()}")
                print(f"\n💡 备份命令:")
                print(f"   !cp -r {dataset_config['model_dir']} {drive_backup_dir}")

        else:
            logger.error(f"❌ 训练失败: {dataset_name}")
            all_metrics[dataset_name] = None

    # 打印最终汇总
    print(f"\n{'='*80}")
    print("训练完成汇总")
    print(f"{'='*80}")

    for dataset_name, metrics in all_metrics.items():
        if metrics:
            storage_loc = "Drive" if metrics.get('storage_location') == 'drive' else "Local"
            print(f"✅ {dataset_name} [{storage_loc}]: {metrics['total_steps']}步, 损失: {metrics['final_loss']:.4f}, 时间: {metrics['training_time']/60:.1f}分钟")
        else:
            print(f"❌ {dataset_name}: 训练失败")

    print(f"{'='*80}")

    # Colab环境特定提示
    if IS_COLAB:
        print("\n📊 训练状态汇总:")
        print(f"   日志文件: {config.LOG_FILE}")
        print(f"   Google Drive结果: {config.RESULTS_DIR}")
        print(f"   Colab本地结果: {config.LOCAL_RESULTS_DIR}")

        print("\n💾 本地模型备份指南:")
        print("   1. 检查本地模型目录:")
        print(f"      !ls -la {config.LOCAL_RESULTS_DIR}")
        print("   2. 备份到Google Drive:")
        print(f"      !mkdir -p {config.RESULTS_DIR}/backups")
        print(f"      !cp -r {config.LOCAL_RESULTS_DIR}/* {config.RESULTS_DIR}/backups/")

        print("\n🔄 恢复训练指南:")
        print("   如果训练被中断，重新运行此脚本时会自动从最新检查点恢复")

    logger.info(f"训练脚本执行完成于: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"\n🏁 训练脚本执行完成于: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ========== 备份函数 ==========
def backup_local_models_to_drive():
    """将本地模型备份到Google Drive"""
    if not IS_COLAB:
        print("此功能仅在Colab环境中可用")
        return

    config = TrainingConfig

    print("开始备份本地模型到Google Drive...")

    # 创建备份目录
    backup_dir = os.path.join(config.RESULTS_DIR, "backups", datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(backup_dir, exist_ok=True)

    # 备份本地模型
    if os.path.exists(config.LOCAL_RESULTS_DIR):
        import shutil
        for item in os.listdir(config.LOCAL_RESULTS_DIR):
            source = os.path.join(config.LOCAL_RESULTS_DIR, item)
            destination = os.path.join(backup_dir, item)

            if os.path.isdir(source):
                try:
                    shutil.copytree(source, destination)
                    print(f"✓ 备份: {item} -> {destination}")
                except Exception as e:
                    print(f"✗ 备份失败 {item}: {e}")
            elif os.path.isfile(source):
                try:
                    shutil.copy2(source, destination)
                    print(f"✓ 备份: {item} -> {destination}")
                except Exception as e:
                    print(f"✗ 备份失败 {item}: {e}")
    else:
        print(f"本地模型目录不存在: {config.LOCAL_RESULTS_DIR}")

    print(f"\n备份完成！备份位置: {backup_dir}")

    # 显示备份内容
    print("\n备份目录内容:")
    !ls -la {backup_dir}

# ========== 脚本入口 ==========
if __name__ == "__main__":
    # 检查是否有备份参数
    if len(sys.argv) > 1 and sys.argv[1] == "--backup":
        backup_local_models_to_drive()
    else:
        try:
            main()
        except Exception as e:
            print(f"\n❌ 错误: {e}")
            import traceback
            traceback.print_exc()

            # 保存错误信息
            try:
                error_file = os.path.join(TrainingConfig.BASE_DIR, "error_log.txt")
                with open(error_file, 'a') as f:  # 追加模式
                    f.write(f"\n{'='*60}\n")
                    f.write(f"错误时间: {datetime.now().isoformat()}\n")
                    f.write(f"错误信息: {str(e)}\n\n")
                    f.write(traceback.format_exc())

                print(f"📝 错误日志已追加到: {error_file}")
            except:
                pass

运行环境: Google Colab


2025-12-22 02:47:54,040 - __main__ - INFO - 训练脚本开始执行 - 混合存储版
INFO:__main__:训练脚本开始执行 - 混合存储版
2025-12-22 02:47:54,047 - __main__ - INFO - Google Drive工作目录: /content/drive/MyDrive/LLM_Impossible_Training
INFO:__main__:Google Drive工作目录: /content/drive/MyDrive/LLM_Impossible_Training
2025-12-22 02:47:54,054 - __main__ - INFO - Colab本地工作目录: /content/LLM_Impossible_Training
INFO:__main__:Colab本地工作目录: /content/LLM_Impossible_Training
2025-12-22 02:47:54,059 - __main__ - INFO - 初始化分词器...
INFO:__main__:初始化分词器...


Mounted at /content/drive
✓ Google Drive 挂载成功
Colab稳健训练脚本 - 不可能语言实验
存储策略：
1. Natural Language: Google Drive
2. Fixed Distance: Google Drive
3. Reversed: Colab本地磁盘
4. Parity Negation: Colab本地磁盘

⚠️  存储位置说明：
   • Natural Language 和 Fixed Distance 模型保存到 Google Drive
   • Reversed 和 Parity Negation 模型保存到 Colab本地磁盘
   • 本地存储的数据在Colab会话结束后会丢失！
   • 如需保留本地模型，请在训练完成后手动复制到Google Drive
创建Google Drive目录:
  ✓ /content/drive/MyDrive/LLM_Impossible_Training
  ✓ /content/drive/MyDrive/LLM_Impossible_Training/data
  ✓ /content/drive/MyDrive/LLM_Impossible_Training/results
  ✓ /content/drive/MyDrive/LLM_Impossible_Training/checkpoints

创建Colab本地目录:
  ✓ /content/LLM_Impossible_Training
  ✓ /content/LLM_Impossible_Training/results
  ✓ /content/LLM_Impossible_Training/checkpoints

创建数据集目录:
  ✓ Natural Language: /content/drive/MyDrive/LLM_Impossible_Training/results/model_natural (Google Drive)
  ✓ Impossible Language (Fixed Distance): /content/drive/MyDrive/LLM_Impossible_Training/results/model_fixed_dist

2025-12-22 02:47:54,492 - __main__ - INFO - 发现已有训练结果: /content/drive/MyDrive/LLM_Impossible_Training/results/model_natural/training_results.json
INFO:__main__:发现已有训练结果: /content/drive/MyDrive/LLM_Impossible_Training/results/model_natural/training_results.json



训练数据集: Natural Language
存储位置: Google Drive


...


✅ 训练完成 - Impossible Language (Reversed):
   存储位置: Colab本地磁盘
   训练步数: 154
   最终损失: 1.8090
   训练时间: 103.22 分钟

⚠️  重要提醒：此模型保存在Colab本地磁盘
   路径: /content/LLM_Impossible_Training/results/model_reversed
   Colab会话结束后这些数据会丢失！
   请在会话结束前手动备份到Google Drive

💡 备份命令:
   !cp -r /content/LLM_Impossible_Training/results/model_reversed /content/drive/MyDrive/LLM_Impossible_Training/results/backup_impossible_language_(reversed)

训练数据集: Impossible Language (Parity Negation)
存储位置: Colab本地磁盘


2025-12-22 04:31:21,372 - __main__ - INFO - 创建新的GPT-2模型
INFO:__main__:创建新的GPT-2模型
/usr/local/lib/python3.12/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
2025-12-22 04:31:22,091 - __main__ - INFO - 数据集加载成功，样本数: 676
INFO:__main__:数据集加载成功，样本数: 676
2025-12-22 04:31:22,318 - __main__ - INFO - 稳健回调初始化，检查点目录: /content/LLM_Impossible_Training/checkpoints/parity_negation
INFO:__main__:稳健回调初始化，检查点目录: /content/LLM_Impossible_Training/checkpoints/parity_negation
2025-12-22 04:31:22,320 - __main__ - INFO - 开始训练过程...
INFO:__main__:开始训练过程...
2025-12-22 04:31:22,322 - __main__ - INFO - ============================================================
INFO:__main__:=========================

Step,Training Loss
1,3.858300
5,3.799800
10,3.679900
15,3.622700
20,3.367700
25,3.201600
30,2.990400
35,2.782700
40,2.632200
45,2.450200


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
2025-12-22 06:24:49,159 - __main__ - INFO - 训练完成: Impossible Language (Parity Negation)
INFO:__main__:训练完成: Impo


✅ 训练完成 - Impossible Language (Parity Negation):
   存储位置: Colab本地磁盘
   训练步数: 170
   最终损失: 1.6370
   训练时间: 113.48 分钟

⚠️  重要提醒：此模型保存在Colab本地磁盘
   路径: /content/LLM_Impossible_Training/results/model_parity_negation
   Colab会话结束后这些数据会丢失！
   请在会话结束前手动备份到Google Drive

💡 备份命令:
   !cp -r /content/LLM_Impossible_Training/results/model_parity_negation /content/drive/MyDrive/LLM_Impossible_Training/results/backup_impossible_language_(parity_negation)

训练完成汇总
✅ Natural Language [Local]: 76步, 损失: 1.8287, 时间: 113.6分钟
✅ Impossible Language (Fixed Distance) [Local]: 95步, 损失: 1.4616, 时间: 114.3分钟
✅ Impossible Language (Reversed) [Local]: 154步, 损失: 1.8090, 时间: 103.2分钟
✅ Impossible Language (Parity Negation) [Local]: 170步, 损失: 1.6370, 时间: 113.5分钟

📊 训练状态汇总:
   日志文件: /content/drive/MyDrive/LLM_Impossible_Training/training.log
   Google Drive结果: /content/drive/MyDrive/LLM_Impossible_Training/results
   Colab本地结果: /content/LLM_Impossible_Training/results

💾 本地模型备份指南:
   1. 检查本地模型目录:
      !ls -la /content/LL

In [10]:
from google.colab import drive
drive.mount('/content/drive')

# 将模型文件夹复制到Google Drive
!cp -r /content/LLM_Impossible_Training/results/model_reversed /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from google.colab import drive
drive.mount('/content/drive')

# 将模型文件夹复制到Google Drive
!cp -r /content/LLM_Impossible_Training/results/model_parity_negation/ /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
